# Model Evaluation 

In [ ]:
# Sample cap: set to an integer (e.g. 5) for quick sample testing, or None for full dataset evaluation
EVAL_MAX_SAMPLES = None  # Set to None for full evaluation, or an integer for subset testing

# Cache tolerance threshold: max allowed missing samples in cache before re-running inference
SAMPLE_COUNT_TOL = 10

# Output & Cache Directories
CACHE_DIR = "out/merged-test"

# Target model configurations:
# - use_val_cache: False (reevaluate val set cleanly and save to val_predictions.pkl)
# - use_test_cache: True (use existing cached predictions in test_predictions.pkl)
TARGET_MODELS = [
    {
        "id": "Baseline",
        "config": "configs/baseline.yaml",
        "checkpoint": "out/baseline_text2cad/checkpoint.pt",
        "folder": "baseline_text2cad",
        "use_val_cache": False,
        "use_test_cache": True,
    },
    {
        "id": "M1",
        "config": "configs/m1.yaml",
        "checkpoint": "out/m1/checkpoint.pt",
        "folder": "m1",
        "use_val_cache": False,
        "use_test_cache": True,
    },
    {
        "id": "M2",
        "config": "configs/m2.yaml",
        "checkpoint": "out/m2/checkpoint.pt",
        "folder": "m2",
        "use_val_cache": False,
        "use_test_cache": True,
    },
    {
        "id": "M3",
        "config": "configs/m3-xx.yaml",
        "checkpoint": "out/the_final_xx/checkpoint.pt",
        "folder": "m3",
        "use_val_cache": False,
        "use_test_cache": False,
    },
    {
        "id": "M4",
        "config": "configs/m4-xx.yaml",
        "checkpoint": "out/m4-xx/checkpoint.pt",
        "folder": "grpo_fuse",
        "use_val_cache": False,
        "use_test_cache": True,
    },
]

# Summary CSV output paths
VAL_SUMMARY_CSV_NAME = "all_5_models_val_evaluation_summary.csv"
TEST_SUMMARY_CSV_NAME = "all_5_models_test_evaluation_summary.csv"


print(f"[Configuration Loaded] EVAL_MAX_SAMPLES = {EVAL_MAX_SAMPLES}")
print(f"[Target Models] {[m['id'] for m in TARGET_MODELS]}")
print(f"[Cache Settings] use_val_cache = False (reevaluating val) | use_test_cache = True (using test cache)")


[Configuration Loaded] EVAL_MAX_SAMPLES = None
[Target Models] ['Baseline', 'M1', 'M2', 'M3', 'M4']
[Cache Settings] use_val_cache = False (reevaluating val) | use_test_cache = True (using test cache)


In [4]:
import os
import sys
import time
import json
import gc
import pickle
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from utils.pipeline import load_config, CustomPipeline
from utils.data_utils import load_val_data, load_test_data
from utils.dual_seq import DualSeq
from utils.representations.CadSeqProc.cad_sequence import CADSequence
from utils.representations.converter import dualseq_to_cadseq
from utils.evaluate import eval_cmd_only
from utils.prediction_cache import save_predictions_pkl, load_predictions_pkl

VAL_SUMMARY_CSV = os.path.join(CACHE_DIR, VAL_SUMMARY_CSV_NAME)
TEST_SUMMARY_CSV = os.path.join(CACHE_DIR, TEST_SUMMARY_CSV_NAME)

print(f"[Configuration Loaded] EVAL_MAX_SAMPLES = {EVAL_MAX_SAMPLES} | Target Models = {[m['id'] for m in TARGET_MODELS]}")
print(f"[Output Paths] VAL: {VAL_SUMMARY_CSV} | TEST: {TEST_SUMMARY_CSV}")

/home/alief/miniconda3/envs/kaggle/lib/python3.12/site-packages/trimesh/grouping.py:15: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.0)
  from scipy.spatial import cKDTree


[Configuration Loaded] EVAL_MAX_SAMPLES = None | Target Models = ['Baseline', 'M1', 'M2', 'M3', 'M4']
[Output Paths] VAL: out/merged-test/all_5_models_val_evaluation_summary.csv | TEST: out/merged-test/all_5_models_test_evaluation_summary.csv


In [5]:
def unify_model_prediction(pred_out, uid: str = "") -> tuple:
    """
    Unified conversion wrapper for baseline (CADSequence) and DualSeq models.
    Returns (DualSeq, CADSequence) tuple safely.
    """
    if pred_out is None:
        return None, None
    if isinstance(pred_out, CADSequence):
        cad_obj = pred_out
        try:
            json_data = cad_obj._json() if hasattr(cad_obj, "_json") else {}
            if json_data and "parts" in json_data:
                ds_obj = DualSeq(json_object=json_data, uid=uid)
            else:
                ds_obj = DualSeq(cmds=[], args=[], uid=uid)
        except Exception:
            ds_obj = DualSeq(cmds=[], args=[], uid=uid)
        return ds_obj, cad_obj
    elif isinstance(pred_out, DualSeq):
        ds_obj = pred_out
        return ds_obj, None
    return None, None

def resolve_cache_path(out_model_dir: str, split_prefix: str) -> str:
    """
    Resolves prediction pkl path strictly to {split_prefix}_predictions.pkl.
    Val predictions are strictly saved to and loaded from val_predictions.pkl.
    Test predictions are strictly saved to and loaded from test_predictions.pkl.
    """
    return os.path.join(out_model_dir, f"{split_prefix}_predictions.pkl")

def validate_and_load_cache(pred_pkl_path: str, samples: list, model_id: str, split_name: str, tolerance: int = SAMPLE_COUNT_TOL) -> dict:
    """
    Validates cache existence and sample coverage for a given split (val/test).
    If missing sample count <= tolerance, proceeds with evaluation without error.
    If missing sample count > tolerance, raises ValueError.
    """
    if not os.path.exists(pred_pkl_path):
        raise FileNotFoundError(f"[Cache Error] Cache file not found at {pred_pkl_path} for model {model_id} ({split_name} cache enabled)")
    
    cached_preds = load_predictions_pkl(pred_pkl_path)
    if not cached_preds:
        raise ValueError(f"[Cache Error] Cache file at {pred_pkl_path} is empty for model {model_id} ({split_name} cache enabled)")
        
    expected_uids = [str(ds.uid) for ds in samples]
    missing_uids = [uid for uid in expected_uids if uid not in cached_preds]
    missing_count = len(missing_uids)
    
    if missing_count > tolerance:
        raise ValueError(
            f"[Cache Error] Cache validation failed for model {model_id} on {split_name} at {pred_pkl_path}! "
            f"Missing {missing_count} / {len(expected_uids)} samples, exceeding tolerance threshold of {tolerance}."
        )
    elif missing_count > 0:
        print(f"[Cache Warning] Model {model_id} ({split_name}): {missing_count} missing samples (within tolerance of {tolerance}). Proceeding with available cached samples.")
    else:
        print(f"[Cache Validated] Successfully loaded all {len(expected_uids)} cached predictions for model {model_id} ({split_name}) from {pred_pkl_path}.")
        
    return cached_preds


In [6]:
print("=== Phase 1: Validation Set Evaluation ===")
val_evaluation_summaries = []

for model_info in TARGET_MODELS:
    model_id = model_info["id"]
    cfg_path = model_info["config"]
    ckpt_path = model_info["checkpoint"]
    folder_name = model_info.get("folder", model_id.lower())
    use_val_cache = model_info.get("use_val_cache", False)
    
    print(f"\n>>> Evaluating Model: {model_id} (folder={folder_name}, use_val_cache={use_val_cache})")
    cfg = load_config(cfg_path)
    pipe = CustomPipeline(cfg)
    pipe.load_tokenizer()
    val_samples = load_val_data(data_folder=cfg.data.data_folder, metadata_csv=cfg.data.metadata_csv, split_json=cfg.data.split_json)
    if EVAL_MAX_SAMPLES is not None:
        val_samples = val_samples[:EVAL_MAX_SAMPLES]
        
    out_model_dir = os.path.join("out", "merged-test", folder_name)
    os.makedirs(out_model_dir, exist_ok=True)
    val_pred_pkl_path = resolve_cache_path(out_model_dir, "val")
    
    if use_val_cache and os.path.exists(val_pred_pkl_path):
        cached_preds = validate_and_load_cache(val_pred_pkl_path, val_samples, model_id, split_name="validation", tolerance=SAMPLE_COUNT_TOL)
    else:
        cached_preds = {}
        
    new_cached_preds = dict(cached_preds)
    missing_samples = [ds for ds in val_samples if str(ds.uid) not in new_cached_preds]
    
    if missing_samples:
        print(f"[Live Inference] Running inference for model {model_id} on {len(missing_samples)} validation samples...")
        pipe.dual_seqs = val_samples
        pipe.val_dual_seqs = val_samples
        pipe._fit_metadata_if_needed()
        pipe.load_model_and_wrapper()
        if ckpt_path and os.path.exists(ckpt_path):
            pipe.load_weights(ckpt_path)
        wrapper = pipe.wrapper
        wrapper.eval()
        
        batch_size = getattr(cfg.data, "batch_size", 32) or 32
        pbar = tqdm(range(0, len(missing_samples), batch_size), desc=f"Val Infer {model_id}")
        for i in pbar:
            chunk = missing_samples[i : i + batch_size]
            chunk_texts = [ds.descriptions.get("expert", "") for ds in chunk]
            with torch.no_grad():
                if hasattr(wrapper, "infer_batch"):
                    chunk_preds = wrapper.infer_batch(chunk_texts, batch_size=len(chunk_texts))
                else:
                    chunk_preds = [wrapper.infer(t) for t in chunk_texts]
            for ds, pred in zip(chunk, chunk_preds):
                new_cached_preds[str(ds.uid)] = pred
                
        save_predictions_pkl(new_cached_preds, val_pred_pkl_path)
        print(f"[Saved Val Cache] Cached {len(new_cached_preds)} validation predictions to: {val_pred_pkl_path}")
        del pipe, wrapper
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    model_preds = []
    for ds in val_samples:
        uid = str(ds.uid)
        gt_cmds = ds.cmds
        gt_args = ds.args_dict
        text = ds.descriptions.get("expert", "")
        pred_out = new_cached_preds.get(uid)
        pred_ds, _ = unify_model_prediction(pred_out, uid=uid)
        model_preds.append({
            "uid": uid,
            "prompt": text,
            "pred_cmds_json": json.dumps(pred_ds.cmds if pred_ds else []),
            "pred_args_json": json.dumps(pred_ds.args_dict if pred_ds else []),
            "gt_cmds_json": json.dumps(gt_cmds),
            "gt_args_json": json.dumps(gt_args),
            "gt_json_obj": json.dumps(ds.json_object if hasattr(ds, "json_object") else {}),
        })
        
    df_preds = pd.DataFrame(model_preds)
    
    eval_rows = []
    pbar_metrics = tqdm(df_preds.iterrows(), total=len(df_preds), desc=f"Val Metrics {model_id}")
    for _, row in pbar_metrics:
        pred_cmds = json.loads(row["pred_cmds_json"])
        pred_args_dict = json.loads(row["pred_args_json"])
        gt_cmds = json.loads(row["gt_cmds_json"])
        gt_args_dict = json.loads(row["gt_args_json"])
        gt_json_obj = json.loads(row["gt_json_obj"]) if "gt_json_obj" in row and pd.notna(row["gt_json_obj"]) else {}
        
        cmd_m = eval_cmd_only(pred_cmds, gt_cmds)
        f1_cmd = cmd_m.get("eval/avg_f1", 0.0)
        correct_args, total_args = 0, 0
        for step_i in range(min(len(pred_args_dict), len(gt_args_dict))):
            p_dict, g_dict = pred_args_dict[step_i], gt_args_dict[step_i]
            for k_arg, v_gt in g_dict.items():
                total_args += 1
                if k_arg in p_dict and abs(float(p_dict[k_arg]) - float(v_gt)) < 1e-2:
                    correct_args += 1
        f1_args = correct_args / max(total_args, 1)
        t2c_macro_f1, t2c_line_f1, t2c_arc_f1, t2c_circle_f1, t2c_extrude_f1 = 0.0, np.nan, np.nan, np.nan, np.nan
        try:
            pred_cad = dualseq_to_cadseq(pred_cmds, pred_args_dict)
            gt_cad = CADSequence.from_minimal_json(gt_json_obj) if gt_json_obj else dualseq_to_cadseq(gt_cmds, gt_args_dict)
            if gt_cad is not None and pred_cad is not None:
                report_df, _ = gt_cad.generate_report(pred_cad, str(row["uid"]))
                if report_df is not None and not report_df.empty:
                    t2c_macro_f1 = float(report_df["macro_f1"].iloc[0])
                    
                    # Average over instances containing at least 1 of the command in ground truth
                    n_line_gt = int(report_df["line_total_gt"].iloc[0]) if "line_total_gt" in report_df.columns else 0
                    if ("LINE" in gt_cmds) or n_line_gt > 0:
                        t2c_line_f1 = float(report_df["line_f1"].iloc[0])
                        
                    n_arc_gt = int(report_df["arc_total_gt"].iloc[0]) if "arc_total_gt" in report_df.columns else 0
                    if ("ARC" in gt_cmds) or n_arc_gt > 0:
                        t2c_arc_f1 = float(report_df["arc_f1"].iloc[0])
                        
                    n_circle_gt = int(report_df["circle_total_gt"].iloc[0]) if "circle_total_gt" in report_df.columns else 0
                    if ("CIRCLE" in gt_cmds) or n_circle_gt > 0:
                        t2c_circle_f1 = float(report_df["circle_f1"].iloc[0])
                        
                    num_ext_gt = float(report_df["num_ext_gt"].iloc[0]) if "num_ext_gt" in report_df.columns else 0.0
                    if any(c.startswith("EXTRUDE_") for c in gt_cmds) or num_ext_gt > 0:
                        b_matches = float(report_df["b"].iloc[0]) if "b" in report_df.columns else 0.0
                        num_ext_pred = float(report_df["num_ext_pred"].iloc[0]) if "num_ext_pred" in report_df.columns else 0.0
                        prec_ext = b_matches / max(num_ext_pred, 1.0)
                        rec_ext = b_matches / max(num_ext_gt, 1.0)
                        t2c_extrude_f1 = 2.0 * prec_ext * rec_ext / (prec_ext + rec_ext) if (prec_ext + rec_ext) > 0 else 0.0
        except Exception:
            pass
        eval_rows.append({
            "f1_cmd": f1_cmd, "f1_args": f1_args, "t2c_macro_f1": t2c_macro_f1, "t2c_line_f1": t2c_line_f1,
            "t2c_arc_f1": t2c_arc_f1, "t2c_circle_f1": t2c_circle_f1, "t2c_extrude_f1": t2c_extrude_f1
        })
    df_eval = pd.DataFrame(eval_rows)
    summary = {
        "Model": model_id,
        "CMD-F1": round(df_eval["f1_cmd"].mean(), 4),
        "ARG-F1": round(df_eval["f1_args"].mean(), 4),
        "T2C-F1": round(df_eval["t2c_macro_f1"].mean(), 4),
        "T2C-Line-F1": round(df_eval["t2c_line_f1"].dropna().mean(), 4) if not df_eval["t2c_line_f1"].dropna().empty else 0.0,
        "T2C-Arc-F1": round(df_eval["t2c_arc_f1"].dropna().mean(), 4) if not df_eval["t2c_arc_f1"].dropna().empty else 0.0,
        "T2C-Circle-F1": round(df_eval["t2c_circle_f1"].dropna().mean(), 4) if not df_eval["t2c_circle_f1"].dropna().empty else 0.0,
        "T2C-Extrude-F1": round(df_eval["t2c_extrude_f1"].dropna().mean(), 4) if not df_eval["t2c_extrude_f1"].dropna().empty else 0.0,
    }
    val_evaluation_summaries.append(summary)
    print(f"Val Evaluated {model_id} -> CMD-F1: {summary['CMD-F1']}, ARG-F1: {summary['ARG-F1']}, T2C-F1: {summary['T2C-F1']}")


=== Phase 1: Validation Set Evaluation ===

>>> Evaluating Model: Baseline (folder=baseline_text2cad, use_val_cache=False)
Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 8925 rows to load JSON payloads...


100%|██████████| 8925/8925 [00:00<00:00, 12282.03it/s]


Successfully created 8921/8925 DualSeq instances.
[Live Inference] Running inference for model Baseline on 8921 validation samples...


The following layers were not sharded: encoder.layer.*.attention.self.value.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.key.weight, embeddings.token_type_embeddings.weight, embeddings.LayerNorm.bias, pooler.dense.weight, encoder.layer.*.attention.output.dense.bias, embeddings.LayerNorm.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.query.weight, embeddings.position_embeddings.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.output.dense.bias, pooler.dense.bias, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.output.LayerNorm.weight


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loaded checkpoint weights from: out/m4/checkpoint.ckpt
Loaded checkpoint weights from: out/m4-xx/checkpoint.pt


Val Infer M4:   0%|          | 0/4461 [00:00<?, ?it/s]

Saved 8921 prediction cache to: out/merged-test/grpo_fuse/val_predictions.pkl
[Saved Val Cache] Cached 8921 validation predictions to: out/merged-test/grpo_fuse/val_predictions.pkl


Val Metrics M4:   0%|          | 0/8921 [00:00<?, ?it/s]

Val Evaluated M4 -> CMD-F1: 0.4395, ARG-F1: 0.7003, T2C-F1: 0.9378


In [7]:
df_val_summary = pd.DataFrame(val_evaluation_summaries)
os.makedirs(os.path.dirname(os.path.abspath(VAL_SUMMARY_CSV)), exist_ok=True)
df_val_summary.to_csv(VAL_SUMMARY_CSV, index=False)
print(f"\nValidation evaluation complete! Summary saved to {VAL_SUMMARY_CSV}")
df_val_summary



Validation evaluation complete! Summary saved to out/merged-test/all_5_models_val_evaluation_summary.csv


,Model,CMD-F1,ARG-F1,T2C-F1,T2C-Line-F1,T2C-Arc-F1,T2C-Circle-F1,T2C-Extrude-F1
0,Baseline,0.3391,0.2107,0.6875,0.5146,0.0494,0.2330,0.7935
1,M1,0.4070,0.1091,0.7652,0.5817,0.0336,0.3015,0.8856
2,M2,0.4014,0.4667,0.8087,0.6200,0.0601,0.3386,0.8915
3,M3,0.4405,0.6986,0.9413,0.7376,0.1643,0.3896,0.9515
4,M4,0.4395,0.7003,0.9378,0.7334,0.1656,0.3887,0.9436


In [8]:
print("=== Phase 2: Official Test Set Evaluation ===")
test_evaluation_summaries = []

for model_info in TARGET_MODELS:
    model_id = model_info["id"]
    cfg_path = model_info["config"]
    ckpt_path = model_info["checkpoint"]
    folder_name = model_info.get("folder", model_id.lower())
    use_test_cache = model_info.get("use_test_cache", True)
    
    print(f"\n>>> Evaluating Model: {model_id} (folder={folder_name}, use_test_cache={use_test_cache})")
    cfg = load_config(cfg_path)
    pipe = CustomPipeline(cfg)
    pipe.load_tokenizer()
    test_samples = load_test_data(data_folder=cfg.data.data_folder, metadata_csv=cfg.data.metadata_csv, split_json=cfg.data.split_json)
    if EVAL_MAX_SAMPLES is not None:
        test_samples = test_samples[:EVAL_MAX_SAMPLES]
        
    out_model_dir = os.path.join("out", "merged-test", folder_name)
    os.makedirs(out_model_dir, exist_ok=True)
    test_pred_pkl_path = resolve_cache_path(out_model_dir, "test")
    
    if use_test_cache and os.path.exists(test_pred_pkl_path):
        cached_preds = validate_and_load_cache(test_pred_pkl_path, test_samples, model_id, split_name="test", tolerance=SAMPLE_COUNT_TOL)
    else:
        cached_preds = {}
        
    new_cached_preds = dict(cached_preds)
    missing_samples = [ds for ds in test_samples if str(ds.uid) not in new_cached_preds]
    
    if missing_samples:
        print(f"[Live Inference] Running inference for model {model_id} on {len(missing_samples)} test samples...")
        pipe.dual_seqs = test_samples
        pipe.val_dual_seqs = test_samples
        pipe._fit_metadata_if_needed()
        pipe.load_model_and_wrapper()
        if ckpt_path and os.path.exists(ckpt_path):
            pipe.load_weights(ckpt_path)
        wrapper = pipe.wrapper
        wrapper.eval()
        
        batch_size = getattr(cfg.data, "batch_size", 32) or 32
        pbar = tqdm(range(0, len(missing_samples), batch_size), desc=f"Test Infer {model_id}")
        for i in pbar:
            chunk = missing_samples[i : i + batch_size]
            chunk_texts = [ds.descriptions.get("expert", "") for ds in chunk]
            with torch.no_grad():
                if hasattr(wrapper, "infer_batch"):
                    chunk_preds = wrapper.infer_batch(chunk_texts, batch_size=len(chunk_texts))
                else:
                    chunk_preds = [wrapper.infer(t) for t in chunk_texts]
            for ds, pred in zip(chunk, chunk_preds):
                new_cached_preds[str(ds.uid)] = pred
                
        save_predictions_pkl(new_cached_preds, test_pred_pkl_path)
        print(f"[Saved Test Cache] Cached {len(new_cached_preds)} test predictions to: {test_pred_pkl_path}")
        del pipe, wrapper
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    model_preds = []
    for ds in test_samples:
        uid = str(ds.uid)
        gt_cmds = ds.cmds
        gt_args = ds.args_dict
        text = ds.descriptions.get("expert", "")
        pred_out = new_cached_preds.get(uid)
        pred_ds, _ = unify_model_prediction(pred_out, uid=uid)
        model_preds.append({
            "uid": uid,
            "prompt": text,
            "pred_cmds_json": json.dumps(pred_ds.cmds if pred_ds else []),
            "pred_args_json": json.dumps(pred_ds.args_dict if pred_ds else []),
            "gt_cmds_json": json.dumps(gt_cmds),
            "gt_args_json": json.dumps(gt_args),
            "gt_json_obj": json.dumps(ds.json_object if hasattr(ds, "json_object") else {}),
        })
        
    df_preds = pd.DataFrame(model_preds)
    
    eval_rows = []
    pbar_metrics = tqdm(df_preds.iterrows(), total=len(df_preds), desc=f"Test Metrics {model_id}")
    for _, row in pbar_metrics:
        pred_cmds = json.loads(row["pred_cmds_json"])
        pred_args_dict = json.loads(row["pred_args_json"])
        gt_cmds = json.loads(row["gt_cmds_json"])
        gt_args_dict = json.loads(row["gt_args_json"])
        gt_json_obj = json.loads(row["gt_json_obj"]) if "gt_json_obj" in row and pd.notna(row["gt_json_obj"]) else {}
        
        cmd_m = eval_cmd_only(pred_cmds, gt_cmds)
        f1_cmd = cmd_m.get("eval/avg_f1", 0.0)
        correct_args, total_args = 0, 0
        for step_i in range(min(len(pred_args_dict), len(gt_args_dict))):
            p_dict, g_dict = pred_args_dict[step_i], gt_args_dict[step_i]
            for k_arg, v_gt in g_dict.items():
                total_args += 1
                if k_arg in p_dict and abs(float(p_dict[k_arg]) - float(v_gt)) < 1e-2:
                    correct_args += 1
        f1_args = correct_args / max(total_args, 1)
        t2c_macro_f1, t2c_line_f1, t2c_arc_f1, t2c_circle_f1, t2c_extrude_f1 = 0.0, np.nan, np.nan, np.nan, np.nan
        try:
            pred_cad = dualseq_to_cadseq(pred_cmds, pred_args_dict)
            gt_cad = CADSequence.from_minimal_json(gt_json_obj) if gt_json_obj else dualseq_to_cadseq(gt_cmds, gt_args_dict)
            if gt_cad is not None and pred_cad is not None:
                report_df, _ = gt_cad.generate_report(pred_cad, str(row["uid"]))
                if report_df is not None and not report_df.empty:
                    t2c_macro_f1 = float(report_df["macro_f1"].iloc[0])
                    
                    # Average over instances containing at least 1 of the command in ground truth
                    n_line_gt = int(report_df["line_total_gt"].iloc[0]) if "line_total_gt" in report_df.columns else 0
                    if ("LINE" in gt_cmds) or n_line_gt > 0:
                        t2c_line_f1 = float(report_df["line_f1"].iloc[0])
                        
                    n_arc_gt = int(report_df["arc_total_gt"].iloc[0]) if "arc_total_gt" in report_df.columns else 0
                    if ("ARC" in gt_cmds) or n_arc_gt > 0:
                        t2c_arc_f1 = float(report_df["arc_f1"].iloc[0])
                        
                    n_circle_gt = int(report_df["circle_total_gt"].iloc[0]) if "circle_total_gt" in report_df.columns else 0
                    if ("CIRCLE" in gt_cmds) or n_circle_gt > 0:
                        t2c_circle_f1 = float(report_df["circle_f1"].iloc[0])
                        
                    num_ext_gt = float(report_df["num_ext_gt"].iloc[0]) if "num_ext_gt" in report_df.columns else 0.0
                    if any(c.startswith("EXTRUDE_") for c in gt_cmds) or num_ext_gt > 0:
                        b_matches = float(report_df["b"].iloc[0]) if "b" in report_df.columns else 0.0
                        num_ext_pred = float(report_df["num_ext_pred"].iloc[0]) if "num_ext_pred" in report_df.columns else 0.0
                        prec_ext = b_matches / max(num_ext_pred, 1.0)
                        rec_ext = b_matches / max(num_ext_gt, 1.0)
                        t2c_extrude_f1 = 2.0 * prec_ext * rec_ext / (prec_ext + rec_ext) if (prec_ext + rec_ext) > 0 else 0.0
        except Exception:
            pass
        eval_rows.append({
            "f1_cmd": f1_cmd, "f1_args": f1_args, "t2c_macro_f1": t2c_macro_f1, "t2c_line_f1": t2c_line_f1,
            "t2c_arc_f1": t2c_arc_f1, "t2c_circle_f1": t2c_circle_f1, "t2c_extrude_f1": t2c_extrude_f1
        })
    df_eval = pd.DataFrame(eval_rows)
    summary = {
        "Model": model_id,
        "CMD-F1": round(df_eval["f1_cmd"].mean(), 4),
        "ARG-F1": round(df_eval["f1_args"].mean(), 4),
        "T2C-F1": round(df_eval["t2c_macro_f1"].mean(), 4),
        "T2C-Line-F1": round(df_eval["t2c_line_f1"].dropna().mean(), 4) if not df_eval["t2c_line_f1"].dropna().empty else 0.0,
        "T2C-Arc-F1": round(df_eval["t2c_arc_f1"].dropna().mean(), 4) if not df_eval["t2c_arc_f1"].dropna().empty else 0.0,
        "T2C-Circle-F1": round(df_eval["t2c_circle_f1"].dropna().mean(), 4) if not df_eval["t2c_circle_f1"].dropna().empty else 0.0,
        "T2C-Extrude-F1": round(df_eval["t2c_extrude_f1"].dropna().mean(), 4) if not df_eval["t2c_extrude_f1"].dropna().empty else 0.0,
    }
    test_evaluation_summaries.append(summary)
    print(f"Test Evaluated {model_id} -> CMD-F1: {summary['CMD-F1']}, ARG-F1: {summary['ARG-F1']}, T2C-F1: {summary['T2C-F1']}")


=== Phase 2: Official Test Set Evaluation ===

>>> Evaluating Model: Baseline (folder=baseline_text2cad, use_test_cache=True)
Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 8046 rows to load JSON payloads...


100%|██████████| 8046/8046 [00:00<00:00, 18372.44it/s]


Successfully created 8042/8046 DualSeq instances.
Loaded 8042 cached predictions from: out/merged-test/baseline_text2cad/test_predictions.pkl
[Cache Validated] Successfully loaded all 8042 cached predictions for model Baseline (test) from out/merged-test/baseline_text2cad/test_predictions.pkl.


Test Metrics Baseline:   0%|          | 0/8042 [00:00<?, ?it/s]

Test Evaluated Baseline -> CMD-F1: 0.3371, ARG-F1: 0.2069, T2C-F1: 0.6857

>>> Evaluating Model: M1 (folder=m1, use_test_cache=True)
Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 8046 rows to load JSON payloads...


...

Test Evaluated M3 -> CMD-F1: 0.4398, ARG-F1: 0.6997, T2C-F1: 0.9412

>>> Evaluating Model: M4 (folder=grpo_fuse, use_test_cache=True)
Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 8046 rows to load JSON payloads...


100%|██████████| 8046/8046 [00:00<00:00, 18131.08it/s]


Successfully created 8042/8046 DualSeq instances.
Loaded 8042 cached predictions from: out/merged-test/grpo_fuse/test_predictions.pkl
[Cache Validated] Successfully loaded all 8042 cached predictions for model M4 (test) from out/merged-test/grpo_fuse/test_predictions.pkl.


Test Metrics M4:   0%|          | 0/8042 [00:00<?, ?it/s]

Test Evaluated M4 -> CMD-F1: 0.439, ARG-F1: 0.7017, T2C-F1: 0.937


In [9]:
df_test_summary = pd.DataFrame(test_evaluation_summaries)
os.makedirs(os.path.dirname(os.path.abspath(TEST_SUMMARY_CSV)), exist_ok=True)
df_test_summary.to_csv(TEST_SUMMARY_CSV, index=False)
print(f"\nTest evaluation complete! Summary saved to {TEST_SUMMARY_CSV}")
df_test_summary



Test evaluation complete! Summary saved to out/merged-test/all_5_models_test_evaluation_summary.csv


,Model,CMD-F1,ARG-F1,T2C-F1,T2C-Line-F1,T2C-Arc-F1,T2C-Circle-F1,T2C-Extrude-F1
0,Baseline,0.3371,0.2069,0.6857,0.5287,0.0517,0.2194,0.7884
1,M1,0.4056,0.0978,0.7657,0.5957,0.0312,0.2900,0.8810
2,M2,0.4008,0.4581,0.8070,0.6316,0.0574,0.3264,0.8906
3,M3,0.4398,0.6997,0.9412,0.7501,0.1609,0.3762,0.9488
4,M4,0.4390,0.7017,0.9370,0.7455,0.1623,0.3754,0.9385


In [ ]:
print("VALIDATION SET SUMMARY TABLE")
print(df_val_summary.to_string(index=False))

print("TEST SET SUMMARY TABLE")
print(df_test_summary.to_string(index=False))


 📋 VALIDATION SET COMPARATIVE SUMMARY TABLE
   Model  CMD-F1  ARG-F1  T2C-F1  T2C-Line-F1  T2C-Arc-F1  T2C-Circle-F1  T2C-Extrude-F1
Baseline  0.3391  0.2107  0.6875       0.5146      0.0494         0.2330          0.7935
      M1  0.4070  0.1091  0.7652       0.5817      0.0336         0.3015          0.8856
      M2  0.4014  0.4667  0.8087       0.6200      0.0601         0.3386          0.8915
      M3  0.4405  0.6986  0.9413       0.7376      0.1643         0.3896          0.9515
      M4  0.4395  0.7003  0.9378       0.7334      0.1656         0.3887          0.9436

 📋 TEST SET COMPARATIVE SUMMARY TABLE
   Model  CMD-F1  ARG-F1  T2C-F1  T2C-Line-F1  T2C-Arc-F1  T2C-Circle-F1  T2C-Extrude-F1
Baseline  0.3371  0.2069  0.6857       0.5287      0.0517         0.2194          0.7884
      M1  0.4056  0.0978  0.7657       0.5957      0.0312         0.2900          0.8810
      M2  0.4008  0.4581  0.8070       0.6316      0.0574         0.3264          0.8906
      M3  0.4398  0.6997  0